# Method Ablations

This notebook runs the three empirical ablations used in the thesis appendix and exports the plots included in the report:

1. zero-order empirical entropy of exponent streams,
2. layer-wise heterogeneity of exponent distributions,
3. exponent n-gram structure.

The model list is read from `examples/config.py` through `MODEL_CONFIG`. Figures are written to `examples/outputs/ablations/figures/`.

In [ ]:
from pathlib import Path
import sys
import math

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

cwd = Path.cwd()
if (cwd / "config.py").exists() and (cwd.parent / "model_compression.py").exists():
    EXAMPLES_DIR = cwd
    ROOT = cwd.parent
elif (cwd / "examples" / "config.py").exists() and (cwd / "model_compression.py").exists():
    ROOT = cwd
    EXAMPLES_DIR = cwd / "examples"
else:
    raise RuntimeError("Run this notebook from Codebase/ or Codebase/examples/.")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(EXAMPLES_DIR))

OUTPUT_DIR = EXAMPLES_DIR / "outputs" / "ablations"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

from config import MODEL_CONFIG
from experiment_utils import (
    collect_named_exponents,
    empirical_entropy_order0,
    pack_non_overlapping_ngrams,
)
from float_utils import get_float_format
from model_compression import HuggingFaceModel

MODEL_CONFIG

## Shared helpers

The helpers below load the configured models, extract exponent streams, and define the metrics used by the three ablations.

In [ ]:
def load_model(spec):
    dtype = spec.get("dtype", "float32")
    return HuggingFaceModel(spec["model_id"], dtype=dtype)


def exponent_width(model):
    first_param = next(model.model.parameters())
    fmt = get_float_format(first_param.dtype)
    return int(fmt.get("exponent_bits", int(fmt["exp_mask"]).bit_length()))


def normalized_histogram(symbols, alphabet_size=256):
    values = symbols.astype(np.uint16, copy=False)
    hist = np.bincount(values, minlength=alphabet_size).astype(np.float64)
    total = hist.sum()
    return hist / total if total else hist


def js_distance(p, q):
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return math.sqrt(0.5 * kl(p, m) + 0.5 * kl(q, m))


def concatenate_exponents(named_exponents):
    parts = [exps.astype(np.uint8, copy=False).reshape(-1) for _, exps in named_exponents if exps.size]
    if not parts:
        return np.array([], dtype=np.uint8)
    return np.concatenate(parts).astype(np.uint8, copy=False)


def ngram_entropy_per_symbol(exponents, n):
    packed = pack_non_overlapping_ngrams(exponents.astype(np.uint8, copy=False), n=n)
    if packed.size == 0:
        return {
            "n": n,
            "complete_ngrams": 0,
            "tail_symbols": int(exponents.size),
            "ngram_entropy_bits": np.nan,
            "entropy_per_exponent_bits": np.nan,
            "unique_ngrams": 0,
        }
    entropy = empirical_entropy_order0(packed)
    usable = packed.size * n
    return {
        "n": n,
        "complete_ngrams": int(packed.size),
        "tail_symbols": int(exponents.size - usable),
        "ngram_entropy_bits": entropy,
        "entropy_per_exponent_bits": entropy / n,
        "unique_ngrams": int(np.unique(packed).size),
    }


def short_label(label, max_len=18):
    return label if len(label) <= max_len else label[: max_len - 1] + "..."


def build_model_runs(model_config):
    runs = []
    load_errors = []
    for spec in model_config:
        label = spec.get("label", spec["model_id"])
        print(f"Loading {label} ({spec['model_id']})")
        try:
            model = load_model(spec)
            named_exponents = collect_named_exponents(model)
            stream = concatenate_exponents(named_exponents)
            runs.append(
                {
                    "label": label,
                    "model_id": spec["model_id"],
                    "dtype": spec.get("dtype", "float32"),
                    "model": model,
                    "named_exponents": named_exponents,
                    "exponents": stream,
                    "exponent_width": exponent_width(model),
                }
            )
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            print(f"Skipping {label}: {message}")
            load_errors.append(
                {
                    "model": label,
                    "model_id": spec["model_id"],
                    "dtype": spec.get("dtype", "float32"),
                    "error": message,
                }
            )
    load_errors_df = pd.DataFrame(load_errors)
    if not load_errors_df.empty:
        load_errors_df.to_csv(OUTPUT_DIR / "load_errors.csv", index=False)
        display(load_errors_df)
    if not runs:
        raise RuntimeError("No configured model could be loaded; check access tokens, model ids, or dependencies.")
    return runs

In [ ]:
runs = build_model_runs(MODEL_CONFIG)
[(run["label"], run["exponents"].size, run["exponent_width"]) for run in runs]

## Ablation 1: zero-order exponent entropy

This ablation checks whether the exponent field is compressible before adding tensor-aware clustering or n-gram packing. A lower empirical entropy than the nominal exponent width means that the exponent stream has statistical redundancy available to a lossless entropy coder.

In [ ]:
zero_order_rows = []
for run in runs:
    exponents = run["exponents"]
    h0 = empirical_entropy_order0(exponents) if exponents.size else np.nan
    width = run["exponent_width"]
    zero_order_rows.append(
        {
            "model": run["label"],
            "model_id": run["model_id"],
            "dtype": run["dtype"],
            "exponent_symbols": int(exponents.size),
            "nominal_exponent_bits": width,
            "h0_bits_per_exponent": h0,
            "entropy_ratio": h0 / width if width and not np.isnan(h0) else np.nan,
            "ideal_savings_bits_per_exponent": width - h0 if not np.isnan(h0) else np.nan,
        }
    )

zero_order_df = pd.DataFrame(zero_order_rows)
display(zero_order_df)

In [ ]:
if not zero_order_df.empty:
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    x = np.arange(len(zero_order_df))
    width = 0.36
    ax.bar(x - width / 2, zero_order_df["nominal_exponent_bits"], width, label="Nominal exponent width")
    ax.bar(x + width / 2, zero_order_df["h0_bits_per_exponent"], width, label="Empirical H0")
    ax.set_xticks(x)
    ax.set_xticklabels([short_label(v) for v in zero_order_df["model"]], rotation=20, ha="right")
    ax.set_ylabel("bits per exponent")
    ax.set_title("Ablation 1: exponent zero-order entropy")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_zero_order_entropy.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

## Ablation 2: layer-wise exponent heterogeneity

This ablation checks whether a single global exponent model is enough. For each tensor, it records zero-order entropy and compares normalized exponent histograms with Jensen-Shannon distance. Non-trivial entropy spread or JS distances support tensor-aware clustering.

In [ ]:
layer_rows = []
heterogeneity_rows = []
histograms_by_model = {}

for run in runs:
    named_exponents = [(name, exps.astype(np.uint8, copy=False).reshape(-1)) for name, exps in run["named_exponents"] if exps.size]
    hists = []
    entropies = []
    for layer_idx, (name, exps) in enumerate(named_exponents):
        h0 = empirical_entropy_order0(exps)
        hist = normalized_histogram(exps)
        entropies.append(h0)
        hists.append(hist)
        layer_rows.append(
            {
                "model": run["label"],
                "layer_idx": layer_idx,
                "tensor": name,
                "symbols": int(exps.size),
                "h0_bits_per_exponent": h0,
            }
        )

    js_values = []
    for i in range(len(hists)):
        for j in range(i + 1, len(hists)):
            js_values.append(js_distance(hists[i], hists[j]))

    histograms_by_model[run["label"]] = hists
    heterogeneity_rows.append(
        {
            "model": run["label"],
            "tensors": len(named_exponents),
            "layer_entropy_mean": float(np.mean(entropies)) if entropies else np.nan,
            "layer_entropy_std": float(np.std(entropies)) if entropies else np.nan,
            "layer_entropy_min": float(np.min(entropies)) if entropies else np.nan,
            "layer_entropy_max": float(np.max(entropies)) if entropies else np.nan,
            "js_mean": float(np.mean(js_values)) if js_values else np.nan,
            "js_median": float(np.median(js_values)) if js_values else np.nan,
            "js_max": float(np.max(js_values)) if js_values else np.nan,
        }
    )

layer_entropy_df = pd.DataFrame(layer_rows)
heterogeneity_df = pd.DataFrame(heterogeneity_rows)
display(heterogeneity_df)
display(layer_entropy_df.sort_values(["model", "h0_bits_per_exponent"]).head(20))

In [ ]:
if not layer_entropy_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    for model_label, group in layer_entropy_df.groupby("model"):
        axes[0].plot(
            group["layer_idx"],
            group["h0_bits_per_exponent"],
            marker=".",
            linewidth=1,
            label=short_label(model_label),
        )
    axes[0].set_title("Layer-wise exponent entropy")
    axes[0].set_xlabel("tensor index")
    axes[0].set_ylabel("H0 bits/exponent")
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)

    x = np.arange(len(heterogeneity_df))
    bar_w = 0.36
    axes[1].bar(x - bar_w / 2, heterogeneity_df["layer_entropy_std"], bar_w, label="entropy std")
    axes[1].bar(x + bar_w / 2, heterogeneity_df["js_mean"], bar_w, label="mean JS")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([short_label(v) for v in heterogeneity_df["model"]], rotation=20, ha="right")
    axes[1].set_title("Heterogeneity summary")
    axes[1].set_ylabel("value")
    axes[1].grid(axis="y", alpha=0.25)
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_layer_heterogeneity.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

## Ablation 3: exponent $n$-gram structure

This ablation tests whether adjacent exponent symbols contain local structure. It compares the entropy of individual exponent symbols with the entropy per original exponent induced by non-overlapping n-grams. If the n-gram entropy per symbol is lower, packed-symbol coding is empirically justified.

In [ ]:
NGRAMS = [2, 3, 4]
NGRAM_MAX_SYMBOLS = 25_000_000
ngram_rows = []

for run in runs:
    exponents = run["exponents"]
    ngram_exponents = exponents[:NGRAM_MAX_SYMBOLS] if exponents.size > NGRAM_MAX_SYMBOLS else exponents
    h0 = empirical_entropy_order0(ngram_exponents) if ngram_exponents.size else np.nan
    for n in NGRAMS:
        stats = ngram_entropy_per_symbol(ngram_exponents, n=n)
        ngram_rows.append(
            {
                "model": run["label"],
                "n": n,
                "sampled_exponent_symbols": int(ngram_exponents.size),
                "symbol_h0_bits_per_exponent": h0,
                "ngram_h0_bits_per_exponent": stats["entropy_per_exponent_bits"],
                "entropy_gain_bits_per_exponent": h0 - stats["entropy_per_exponent_bits"] if not np.isnan(stats["entropy_per_exponent_bits"]) else np.nan,
                "complete_ngrams": stats["complete_ngrams"],
                "tail_symbols": stats["tail_symbols"],
                "unique_ngrams": stats["unique_ngrams"],
            }
        )

ngram_df = pd.DataFrame(ngram_rows)
display(ngram_df)

In [ ]:
if not ngram_df.empty:
    heatmap_df = (
        ngram_df.assign(gain_mbits=lambda df: df["entropy_gain_bits_per_exponent"] * 1000.0)
        .pivot(index="model", columns="n", values="gain_mbits")
        .loc[list(ngram_df["model"].drop_duplicates())]
    )
    values = heatmap_df.to_numpy(dtype=float)
    positive = values[np.isfinite(values) & (values > 0)]
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    if positive.size:
        norm = matplotlib.colors.LogNorm(vmin=max(float(positive.min()), 1e-3), vmax=float(positive.max()))
        image = ax.imshow(values, cmap="viridis", norm=norm, aspect="auto")
    else:
        image = ax.imshow(values, cmap="viridis", aspect="auto")
    for row_idx, model_label in enumerate(heatmap_df.index):
        for col_idx, n in enumerate(heatmap_df.columns):
            value = values[row_idx, col_idx]
            if not np.isfinite(value):
                label = "n/a"
                text_color = "white"
            elif value < 10:
                label = f"{value:.2f}"
                text_color = "black" if image.norm(value) > 0.7 else "white"
            else:
                label = f"{value:.1f}"
                text_color = "black" if image.norm(value) > 0.7 else "white"
            ax.text(col_idx, row_idx, label, ha="center", va="center", color=text_color, fontsize=10)
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels([f"n={n}" for n in heatmap_df.columns])
    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels([short_label(v, max_len=28) for v in heatmap_df.index])
    ax.set_title("Ablation 3: exponent n-gram entropy gain")
    ax.set_xlabel("packed exponent group size")
    ax.set_ylabel("model")
    cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("gain (millibits/exponent, log scale)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "ablation_ngram_entropy_gain.png", dpi=220, bbox_inches="tight")
    display(fig)
    plt.close(fig)

## Export tables

The CSV files below can be copied into the thesis results or used to regenerate appendix tables and plots.

In [ ]:
zero_order_df.to_csv(OUTPUT_DIR / "zero_order_entropy.csv", index=False)
heterogeneity_df.to_csv(OUTPUT_DIR / "layer_heterogeneity_summary.csv", index=False)
layer_entropy_df.to_csv(OUTPUT_DIR / "layer_entropy.csv", index=False)
ngram_df.to_csv(OUTPUT_DIR / "ngram_entropy.csv", index=False)

print(f"Tables written to: {OUTPUT_DIR}")
print(f"Figures written to: {FIGURE_DIR}")
list(FIGURE_DIR.glob("*.png"))